# 7.12 · 概念漂移 / Concept Drift

> **课程定位 / Where this fits**
> 第 12 课，**Part 7 · 模型评估与优化**（Part 7 收官）。
> Lesson 12, **Part 7 · Model Evaluation & Tuning** (Part 7 finale).
>
> 模型上线那一刻是它最准的时候——之后只会变差。因为真实世界**在变**：用户偏好变、欺诈手法变、经济环境变。当"输入与输出的关系"随时间改变，模型就**悄悄失效**，这叫**概念漂移**。它是 MLOps 的核心问题：你得**检测到漂移**并**触发重训**。这一课从零实现漂移检测(DDM)并演示滑动窗口重训。
> A model is at its best the moment it ships — only downhill from there, because the world **changes**: user tastes, fraud tactics, the economy. When the input-output relationship shifts over time, the model **silently decays** — this is **concept drift**. It's a core MLOps problem: you must **detect drift** and **trigger retraining**. This lesson implements drift detection (DDM) from scratch and demonstrates sliding-window retraining.
>
> 💼 **实战/面试视角**："模型上线后怎么监控 / 什么是概念漂移 / 怎么检测和应对" 偏 MLOps/生产 ML。
> 💼 **Practical/interview angle:** "monitoring after deployment / what is concept drift / detect and respond" — MLOps/production ML.

> 💡 **面试相关 / Interview-relevant**
> - "什么是概念漂移 / 和数据漂移的区别"（出镜率 ★★★★）
> - "怎么检测漂移（监控指标/分布/DDM/ADWIN）"（★★★★）
> - "检测到漂移怎么办（重训/滑窗/在线学习）"（★★★★）
> - "漂移的几种类型（突变/渐变/季节）"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解概念漂移 vs 数据漂移，及漂移的类型。
   Understand concept vs data drift, and types of drift.
2. 看固定模型在漂移流上如何悄悄退化。
   See a frozen model silently decay on a drifting stream.
3. **从零**实现 DDM 漂移检测器。
   Implement a DDM drift detector from scratch.
4. 用**滑动窗口重训**应对漂移。
   Respond with sliding-window retraining.

## 目录 / TOC
1. [先建直觉 + 漂移类型 ⭐](#1)
2. [🌊 数据：会漂移的数据流](#2)
3. [固定模型悄悄退化 ⭐](#3)
4. [DDM 漂移检测（从零）⭐](#4)
5. [应对：滑动窗口重训 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + 漂移类型 ⭐ / Intuition & Types of Drift

模型假设"未来和过去同分布"，但生产环境会打破它。区分两个概念（面试要点）：
A model assumes "the future is distributed like the past", but production breaks this. Two concepts (interview points):
- **数据漂移(data/covariate drift)**：输入 $\Pr(\mathbf{x})$ 的分布变了（如新用户群体涌入），但"x→y 的规律"没变。
  **Data (covariate) drift:** the input distribution $\Pr(\mathbf{x})$ changes (e.g. a new user segment), but the x→y rule is unchanged.
- **概念漂移(concept drift)**：$\Pr(y\mid\mathbf{x})$ 变了——**同样的输入，正确答案变了**（如疫情后"出行频繁"不再代表高消费能力）。这更致命，因为模型学的规律本身过时了。
  **Concept drift:** $\Pr(y\mid\mathbf{x})$ changes — **same input, different correct answer** (e.g. post-pandemic, "travels often" no longer means high spending). More fatal, because the learned rule itself is stale.

漂移的几种形态：**突变(sudden)**（一夜之间变，如政策改变）、**渐变(gradual/incremental)**（缓慢演变，如用户口味）、**季节性(recurring)**（周期回归，如节假日）。检测策略要匹配形态。
Forms of drift: **sudden** (overnight, e.g. a policy change), **gradual/incremental** (slow evolution, e.g. tastes), **recurring/seasonal** (cyclic, e.g. holidays). The detection strategy should match the form.


<a id="2"></a>
## 2. 数据：会漂移的数据流 / A Drifting Data Stream

造一个**流式分类任务**：前一半数据用规律 A 生成，某个时刻**突变**成规律 B（$\Pr(y\mid\mathbf{x})$ 翻转）——模拟概念漂移。数据**按时间顺序**到达（所以绝不能随机划分，3.10）。
We build a **streaming classification task**: the first half follows rule A, then at some point it **suddenly switches** to rule B ($\Pr(y\mid\mathbf{x})$ flips) — simulating concept drift. Data arrives **in time order** (so never random-split, 3.10).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import SGDClassifier
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

n = 4000
drift_point = 2000                                    # 概念漂移发生的时刻 / when concept changes
X = rng.normal(0, 1, (n, 2))
y = np.zeros(n, dtype=int)
# 规律 A(前半): y 由 x0 决定 / rule A: y depends on x0
y[:drift_point] = (X[:drift_point, 0] > 0).astype(int)
# 规律 B(后半, 突变): y 改由 x1 决定(同样的x, 答案变了=概念漂移) / rule B: y now depends on x1
y[drift_point:] = (X[drift_point:, 1] > 0).astype(int)
print(f"流式数据 {n} 条; 在第 {drift_point} 条发生概念漂移(规律从'看x0'突变为'看x1')")
print("这是概念漂移: P(y|x) 变了 — 同样的输入, 正确答案变了")


<a id="3"></a>
## 3. 固定模型悄悄退化 ⭐ / A Frozen Model Silently Decays

最危险的情形：模型训练好就**冻结不动**，没人盯着。在漂移点之前它很准，漂移后**准确率断崖式下跌**，但如果你不监控，根本不会察觉——线上还在用一个已经失效的模型。下面用滑动窗口准确率把这个"悄悄退化"画出来。
The most dangerous scenario: train a model, **freeze it**, nobody watches. Before the drift it's accurate; after, accuracy **falls off a cliff** — but without monitoring you'd never notice, serving a dead model in production. We plot this "silent decay" via a rolling accuracy.


In [ ]:
# 只用漂移前的数据训练, 然后冻结 / train on pre-drift data only, then freeze
frozen = SGDClassifier(loss="log_loss", random_state=0)
frozen.fit(X[:1500], y[:1500])

# 在整条流上逐点预测(模拟上线后), 算滑动窗口准确率 / rolling accuracy over the stream
correct = (frozen.predict(X) == y).astype(float)
window = 200
rolling = np.convolve(correct, np.ones(window)/window, mode="valid")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(window-1, n), rolling, lw=1.5)
ax.axvline(drift_point, color="r", ls="--", label="概念漂移点 drift point")
ax.axhline(0.5, color="gray", ls=":", label="随机基线 chance")
ax.set_xlabel("时间(数据流位置)"); ax.set_ylabel(f"滑动窗口准确率(窗口={window})")
ax.set_title("冻结模型: 漂移后准确率断崖下跌 — 不监控就发现不了")
ax.legend(); plt.tight_layout(); plt.show()
print(f"漂移前准确率 ≈ {rolling[:drift_point-window].mean():.2f}, 漂移后 ≈ {rolling[drift_point:].mean():.2f}")
print("→ 模型悄悄从'很准'退化到'接近随机'; 没有监控你根本不知道")


<a id="4"></a>
## 4. DDM 漂移检测（从零）⭐ / DDM Drift Detection

怎么自动检测到漂移？**DDM（Drift Detection Method）** 是最经典的方法之一，思路简单：**持续监控模型的错误率**。如果数据稳定，错误率应该稳定或下降；一旦错误率**显著上升**（超过历史最低水平若干个标准差），就报警。
How to detect drift automatically? **DDM (Drift Detection Method)** is a classic, with a simple idea: **continuously monitor the error rate**. On stable data, error stays flat or drops; once it **rises significantly** (above the historical minimum by several standard deviations), raise an alarm.

具体：记录至今的错误率 $p$ 和其标准差 $s=\sqrt{p(1-p)/i}$，追踪历史最小的 $p+s$。当当前 $p+s$ 超过 $\min(p+s) + 2s_{\min}$ → **警告(warning)**；超过 $+3s_{\min}$ → **漂移(drift)**，触发重训。这就是从零实现的核心。
Specifically: track the running error rate $p$ and its std $s=\sqrt{p(1-p)/i}$, and the historical minimum of $p+s$. When current $p+s$ exceeds $\min(p+s) + 2s_{\min}$ → **warning**; exceeds $+3s_{\min}$ → **drift**, triggering retraining. That's the from-scratch core.


In [ ]:
class DDM:
    def __init__(self, warn_level=2.0, drift_level=3.0, min_samples=100):
        self.warn_level, self.drift_level, self.min_samples = warn_level, drift_level, min_samples
        self.n = 0; self.p = 1.0; self.s = 0.0
        self.p_min = float("inf"); self.s_min = float("inf")
    def update(self, error):                              # error: 这次预测是否出错(1/0)
        self.n += 1
        self.p += (error - self.p) / self.n              # 增量更新错误率均值 / running error rate
        self.s = (self.p * (1 - self.p) / self.n) ** 0.5  # 其标准差(二项)
        if self.n < self.min_samples:
            return "ok"
        if self.p + self.s < self.p_min + self.s_min:    # 刷新历史最低(p+s) / track the best so far
            self.p_min, self.s_min = self.p, self.s
        if self.p + self.s > self.p_min + self.drift_level * self.s_min:
            return "drift"                               # 错误率显著超历史最低 → 漂移
        if self.p + self.s > self.p_min + self.warn_level * self.s_min:
            return "warning"
        return "ok"

# 在冻结模型的错误流上跑 DDM / run DDM on the frozen model's error stream
ddm = DDM()
errors = (frozen.predict(X) != y).astype(int)
first_drift = None
for i, e in enumerate(errors):
    st = ddm.update(e)
    if st == "drift" and first_drift is None:
        first_drift = i; break
print(f"真实漂移点: {drift_point}")
print(f"DDM 检测到漂移: 第 {first_drift} 条 (滞后 {first_drift - drift_point} 条 = 检测延迟)")
print("→ DDM 在漂移后不久就报警, 可自动触发重训(检测延迟是这类方法的关键指标)")


<a id="5"></a>
## 5. 应对：滑动窗口重训 + 小结 ⭐ / Response & Summary

检测到漂移只是第一步，还要**应对**。三种常见策略：
Detection is step one; you must also **respond**. Three common strategies:
- **触发式重训**：DDM 报警 → 用最近的数据重训模型。
  **Triggered retraining:** DDM alarms → retrain on recent data.
- **定期重训**：不管检不检测，按固定周期（每天/每周）重训——简单可靠。
  **Scheduled retraining:** retrain on a fixed cadence regardless — simple and reliable.
- **在线/增量学习**：模型用 `partial_fit` 持续吸收新数据（接 5.15），自然适应漂移。
  **Online/incremental learning:** the model keeps absorbing new data via `partial_fit` (5.15), adapting naturally.

下面对比"冻结模型"和"滑动窗口持续重训"在漂移流上的表现——后者在漂移后能**快速恢复**。
Below we compare a frozen model vs a sliding-window-retrained model on the drifting stream — the latter **recovers quickly** after drift.


In [ ]:
# 滑动窗口: 每来一批新数据就用'最近 win 条'重训 / retrain on the most recent window
win, step = 500, 100
adaptive_acc, frozen_acc, centers = [], [], []
for start in range(win, n - step, step):
    Xw, yw = X[start-win:start], y[start-win:start]      # 最近 win 条作训练
    Xb, yb = X[start:start+step], y[start:start+step]    # 下一批作测试(模拟未来)
    m = SGDClassifier(loss="log_loss", random_state=0).fit(Xw, yw)
    adaptive_acc.append((m.predict(Xb) == yb).mean())
    frozen_acc.append((frozen.predict(Xb) == yb).mean())
    centers.append(start)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(centers, frozen_acc, "o-", label="冻结模型 frozen", alpha=0.7)
ax.plot(centers, adaptive_acc, "s-", label="滑动窗口重训 sliding-window", alpha=0.7)
ax.axvline(drift_point, color="r", ls="--", label="漂移点 drift")
ax.set_xlabel("时间"); ax.set_ylabel("对'下一批'的准确率"); ax.legend()
ax.set_title("应对漂移: 冻结模型崩溃不恢复; 滑窗重训快速适应新规律")
plt.tight_layout(); plt.show()
print(f"漂移后平均准确率: 冻结={np.mean([a for c,a in zip(centers,frozen_acc) if c>drift_point]):.2f}, "
      f"滑窗重训={np.mean([a for c,a in zip(centers,adaptive_acc) if c>drift_point]):.2f}")
print("→ 滑窗重训在漂移后快速恢复; 这就是'检测漂移→重训'闭环的价值")


```
概念漂移: P(y|x) 变了(同样输入答案变); 数据漂移: P(x) 变了(输入分布变)
形态: 突变(政策)/渐变(口味)/季节(节假日); 检测策略要匹配
冻结模型在漂移后悄悄退化到接近随机 — 不监控发现不了 (生产 ML 的核心风险)
检测: 监控滑动窗口指标; DDM(监控错误率, 超历史最低 +2s 警告 +3s 漂移); ADWIN(自适应窗口)
应对: 触发式重训(检测到就重训) / 定期重训 / 在线学习(partial_fit, 5.15)
闭环: 监控→检测漂移→重训; 滑窗重训能在漂移后快速恢复
```

### 💡 面试速查 / Interview cheat-sheet
1. **概念漂移 P(y|x) 变 vs 数据漂移 P(x) 变**; 前者更致命。
   Concept drift changes P(y|x); data drift changes P(x); the former is worse.
2. **冻结模型会悄悄退化** → 必须监控(生产 ML 核心风险)。
   Frozen models silently decay → must monitor (core production-ML risk).
3. **检测**: 监控指标/分布; DDM(错误率)/ADWIN(自适应窗口)。
   Detect by monitoring metrics/distributions; DDM (error rate) / ADWIN (adaptive window).
4. **应对**: 触发重训 / 定期重训 / 在线学习(partial_fit)。
   Respond: triggered retraining / scheduled retraining / online learning.
5. **检测延迟**是漂移检测器的关键指标(越快越好)。
   Detection delay is the key metric for a drift detector (shorter is better).

### Part 7 完成 🎉
评估与优化全部走通: 回归/分类指标 → 偏差方差 → 交叉验证 → 调参 → 多目标 → 特征选择 → 可解释性 → 校准 → 公平性 → 鲁棒性 → 概念漂移。这些是把模型"做出来"变成"用得好、信得过、撑得住"的关键能力。
Part 7 complete: regression/classification metrics → bias-variance → CV → tuning → multi-objective → feature selection → interpretability → calibration → fairness → robustness → concept drift — the skills that turn a built model into one that's usable, trustworthy, and durable.
